#Import

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

#Read Bronze Layer

In [0]:
df = spark.table("workspace.bronze.erp_loc_a101")

#Silver Transformations

In [0]:
df.limit(10).display()

##Trimming

In [0]:
for filed in df.schema.fields:
    if isinstance(filed.dataType,StringType):
        df = df.withColumn(filed.name,trim(col(filed.name)))

##Customer ID Cleanup

In [0]:
df = df.withColumn("cid",regexp_replace(col("cid"),"-",""))

##Country Normalization

In [0]:
# Standardize country codes and handle missing country values.
df = df.withColumn(
    "cntry",
    when(upper(col("cntry")) == "DE", "Germany")
    .when(upper(col("cntry")).isin("US", "USA"), "United States")
    .when((col("cntry") == "") | (col("cntry").isNull()), "n/a")
    .otherwise(col("cntry"))
)



##Renaming Columns

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "cntry": "country"
}

for old_name , new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name,new_name)


##Sanity checks of dataframe

In [0]:
df.limit(10).display()

#Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customer_location")

#Sanity checks of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.erp_customer_location